# 电商商品知识与售后 RAG · 8B
在 Kaggle 编辑页开启 GPU，依次运行下面两格完成启动。第一次需要重建 147,702 条商品的检索库。接下来运行任意 `ask("你的问题")`，同一会话可以继续追问或换话题。商品事实回答会列出证据。首次照片查询按需读取官方图源，需要可访问的网络。

In [ ]:
from pathlib import Path
import runpy
import sys
from zipfile import ZipFile

inputs = Path("/kaggle/input")
markers = list(inputs.rglob("ABO_8B_FORMAL.txt", recurse_symlinks=True))
if markers:
    marker, = markers
    entry = marker.parent / "scripts/kaggle_8b_project.py"
else:
    bundle, = inputs.rglob("abo-8b-formal.zip", recurse_symlinks=True)
    with ZipFile(bundle) as archive:
        entry = Path(archive.extract("scripts/kaggle_8b_project.py", "/tmp/abo-rag-bootstrap"))
bootstrap = runpy.run_path(str(entry))
bootstrap["setup"]()
root = bootstrap["ROOT"]
if not (root / "data/ecommerce_rag/abo-full-v1/catalog.sqlite").is_file():
    bootstrap["setup_catalog"]()
else:
    import os

    os.environ["ABO_FONT_PATH"] = str(root / "fonts/NotoSansSC.ttf")
code = str(bootstrap["CODE"])
if code not in sys.path:
    sys.path.insert(0, code)

In [ ]:
from full_app import FullApplication

if "app" in globals():
    app.close()
app = FullApplication("B", mode="bm25", model="8b")
session = app.session()

def ask(question):
    result = session.respond(question)
    print(result["answer"])
    if result["citations"]:
        print("证据：" + "、".join(result["citations"]))

print("Qwen3-VL-8B-Instruct 已就绪。每轮记录保存在", session.out)

In [ ]:
ask("搜索 黑色双肩包")

In [ ]:
ask("第一个")

In [ ]:
ask("这个商品的品牌是什么？")

In [ ]:
ask("换个话题，用一句话说说月亮为什么有圆缺")

在新单元格输入其他商品或任意话题。可以用 `ask("目录")` 浏览商品类目，或直接用 ASIN@站点 指定商品。导出记录时查看 `session.out` 下的 `对话记录.txt`、`逐题结果.txt` 和 `results.jsonl`。